# Hiring Pipeline Optimization — Interactive Exploration

Run each cell top to bottom. By the end you'll see:
- Historical data overview: hire rate, outcome distribution, stage funnel
- How precision, recall, and cost evolve as the algorithm learns
- How each stage's threshold converges over time
- What difference `cost_weight` makes

In [ ]:
# Colab setup — clones the repo if it doesn't exist locally
import os
if not os.path.isdir('src') and not os.path.isdir('../src'):
    os.system('git clone https://github.com/fatisati/hiring-bandit.git')
    os.chdir('hiring-bandit')

In [ ]:
import sys
import os

# works from repo root (Colab) or notebooks/ subfolder (local)
src_dir = 'src' if os.path.isdir('src') else os.path.join('..', 'src')
sys.path.insert(0, src_dir)

import matplotlib.pyplot as plt
import pandas as pd

from generate_data import generate_dataset, MEAN_PERFORMANCE
from bandit import HiringPipeline
from evaluate import Evaluator

STAGE_COSTS = {'s0': 1, 's1': 3, 's2': 8, 's3': 15}

## 1. Generate Data

200 candidates. First 50 are historical (warm start). Remaining 150 are online.

In [ ]:
data      = generate_dataset(n=200, seed=42)
historical = [c for c in data if c['phase'] == 'historical']
online     = [c for c in data if c['phase'] == 'online']

df_hist = pd.DataFrame(historical)

hired      = df_hist['hired'].sum()
not_hired  = len(df_hist) - hired
truly_good = df_hist['ground_truth_hire'].sum()

print(f"Historical candidates : {len(df_hist)}")
print(f"Truly good (top N)    : {truly_good} ({100*truly_good//len(df_hist)}%)")
print(f"Hired                 : {hired} ({100*hired//len(df_hist)}%)")

df_hist[['candidate_id', 'true_quality', 's0_score', 's1_score', 's2_score', 's3_score',
         'ground_truth_hire', 'hired', 'outcome']].head(8)

In [ ]:
stages = ['s0', 's1', 's2', 's3']
fig, axes = plt.subplots(1, len(stages), figsize=(16, 4))
fig.suptitle('Score Distribution per Stage — green = truly good, red = not', fontsize=13)

for ax, stage in zip(axes, stages):
    scores = df_hist[[f'{stage}_score', 'ground_truth_hire']].dropna()
    good = scores[scores['ground_truth_hire'] == 1][f'{stage}_score']
    bad  = scores[scores['ground_truth_hire'] == 0][f'{stage}_score']
    ax.hist(bad,  bins=15, color='#d9534f', alpha=0.6, label='Not truly good')
    ax.hist(good, bins=15, color='#5cb85c', alpha=0.8, label='Truly good')
    ax.set_title(f'{stage}  (n={len(scores)})')
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

## 2. Run the Pipeline

Warm starts on historical data, then processes online candidates one by one.
Thresholds are recorded after every candidate so we can plot convergence.

In [ ]:
def run_pipeline(historical, online, exploration=1.0, cost_weight=0.1, batch_size=10):
    pipeline  = HiringPipeline(stage_costs=STAGE_COSTS, exploration=exploration, cost_weight=cost_weight)
    evaluator = Evaluator(batch_size=batch_size)

    pipeline.warm_start(historical)

    threshold_history = {s: [] for s in pipeline.stages}

    for candidate in online:
        result  = pipeline.process(candidate)
        reward  = pipeline.compute_reward(result, candidate['outcome'])
        pipeline.update(result['thresholds'], result['visited_stages'], reward)
        evaluator.record(candidate, result)

        for stage in pipeline.stages:
            threshold_history[stage].append(pipeline.bandits[stage].select_threshold())

    return evaluator, threshold_history


evaluator, threshold_history = run_pipeline(historical, online)
evaluator.print_report()

## 3. Metrics Over Time

A well-functioning algorithm shows precision and recall trending up, cost trending down.

In [ ]:
batches       = [m.batch         for m in evaluator.batches]
precision     = [m.precision     for m in evaluator.batches]
recall        = [m.recall        for m in evaluator.batches]
cost_per_hire = [m.cost_per_hire for m in evaluator.batches]
total_cost    = [m.total_cost    for m in evaluator.batches]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Algorithm Performance Over Time', fontsize=14)

axes[0, 0].plot(batches, precision, marker='o', color='steelblue')
axes[0, 0].set_title('Precision')
axes[0, 0].set_xlabel('Batch')
axes[0, 0].set_ylim(0, 1.05)
axes[0, 0].axhline(y=1.0, linestyle='--', color='gray', alpha=0.4)

axes[0, 1].plot(batches, recall, marker='o', color='darkorange')
axes[0, 1].set_title('Recall')
axes[0, 1].set_xlabel('Batch')
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].axhline(y=1.0, linestyle='--', color='gray', alpha=0.4)

axes[1, 0].plot(batches, cost_per_hire, marker='o', color='crimson')
axes[1, 0].set_title('Cost per Hire (hours)')
axes[1, 0].set_xlabel('Batch')

axes[1, 1].plot(batches, total_cost, marker='o', color='purple')
axes[1, 1].set_title('Total Cost per Batch (hours)')
axes[1, 1].set_xlabel('Batch')

plt.tight_layout()
plt.show()

## 4. Threshold Convergence

Each line shows the threshold the algorithm is currently using for that stage.
You should see it stabilise after enough candidates.

In [ ]:
stages = list(STAGE_COSTS.keys())
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Threshold Convergence per Stage', fontsize=14)

for i, stage in enumerate(stages):
    ax = axes[i // 2, i % 2]
    ax.plot(threshold_history[stage], color='steelblue', alpha=0.8)
    ax.set_title(f'Stage {stage}')
    ax.set_xlabel('Online candidate')
    ax.set_ylabel('Threshold')
    ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 5. Cost Weight Comparison

`cost_weight=0` — cost blind, only cares about hire quality  
`cost_weight=0.1` — penalises hours spent, pushes the algorithm to reject earlier

In [ ]:
ev_no_cost, _ = run_pipeline(historical, online, cost_weight=0.0)
ev_cost, _    = run_pipeline(historical, online, cost_weight=0.1)

b = [m.batch for m in ev_no_cost.batches]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('cost_weight=0.0 vs cost_weight=0.1', fontsize=13)

for ax, metric, label, color_pair in zip(
    axes,
    ['precision', 'recall', 'cost_per_hire'],
    ['Precision', 'Recall', 'Cost per Hire (hrs)'],
    [('steelblue', 'navy'), ('darkorange', 'saddlebrown'), ('crimson', 'darkred')]
):
    vals_0  = [getattr(m, metric) for m in ev_no_cost.batches]
    vals_01 = [getattr(m, metric) for m in ev_cost.batches]
    ax.plot(b, vals_0,  marker='o', label='cost_weight=0.0', color=color_pair[0])
    ax.plot(b, vals_01, marker='s', label='cost_weight=0.1', color=color_pair[1], linestyle='--')
    ax.set_title(label)
    ax.set_xlabel('Batch')
    ax.legend(fontsize=8)
    if metric in ('precision', 'recall'):
        ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    'metric': ['avg precision', 'avg recall', 'avg cost/hire', 'total cost'],
    'cost_weight=0.0': [
        f"{sum(m.precision     for m in ev_no_cost.batches) / len(ev_no_cost.batches):.2f}",
        f"{sum(m.recall        for m in ev_no_cost.batches) / len(ev_no_cost.batches):.2f}",
        f"{sum(m.cost_per_hire for m in ev_no_cost.batches) / len(ev_no_cost.batches):.1f} hrs",
        f"{sum(m.total_cost    for m in ev_no_cost.batches):.0f} hrs",
    ],
    'cost_weight=0.1': [
        f"{sum(m.precision     for m in ev_cost.batches) / len(ev_cost.batches):.2f}",
        f"{sum(m.recall        for m in ev_cost.batches) / len(ev_cost.batches):.2f}",
        f"{sum(m.cost_per_hire for m in ev_cost.batches) / len(ev_cost.batches):.1f} hrs",
        f"{sum(m.total_cost    for m in ev_cost.batches):.0f} hrs",
    ],
})
summary